# Donor peptide odds ratios


In [ ]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy.stats import fisher_exact
from statsmodels.stats.contingency_tables import Table2x2
from scipy import sparse
import statsmodels.api as sm

In [ ]:
H5AD_PATH = "data/adata_cohort1.h5ad"

ZSCORE_LAYER = "zscores_4andhalf"
ZSCORE_THRESHOLD = 4.5

GROUP_COL = "group"
CASE_LABEL = "SLE"
CONTROL_LABEL = "healthy_control"
DONOR_COL = "unique_patient_id"

MIN_DONOR_COUNT = 0

In [ ]:
def load_peptide_mapping(path):
    adata = ad.read_h5ad(path, backed="r")
    peptide_gene_mapping = pd.DataFrame({
        'seq_id': adata.var_names.astype(str),
        'gene': adata.var['gene'].astype(str).values,
    })
    adata.file.close()
    peptide_gene_mapping['fragment'] = (
        peptide_gene_mapping['seq_id'].str.extract(r'fragment_(\d+)').astype(int)
    )
    peptide_gene_mapping['isoform'] = (
        peptide_gene_mapping['seq_id'].str.extract(r'isoform_(\d+)')
    )

    gene_counts = peptide_gene_mapping.groupby('gene').size()
    single_peptide_genes = gene_counts[gene_counts == 1].index

    peptide_gene_mapping['isoform'] = np.where(
        peptide_gene_mapping['gene'].isin(single_peptide_genes),
        np.nan, 'iso' + peptide_gene_mapping['isoform'].fillna('')
    )
    peptide_gene_mapping['isoform'] = peptide_gene_mapping['isoform'].replace('iso', np.nan)

    peptide_gene_mapping['gene_fragment'] = (
        peptide_gene_mapping['gene'] + '_' + peptide_gene_mapping['fragment'].astype(str)
    )
    peptide_gene_mapping['pep_short'] = np.where(
        peptide_gene_mapping['isoform'].isna(),
        peptide_gene_mapping['gene_fragment'],
        peptide_gene_mapping['gene_fragment'] + '_' + peptide_gene_mapping['isoform']
    )
    return peptide_gene_mapping

def get_donor_index_and_case_status(adata, donor_col, group_col, control_label):
    if donor_col not in adata.obs.columns:
        raise ValueError(f"{donor_col} not found in adata.obs")

    df = adata.obs[[donor_col, group_col]].copy()
    df[donor_col] = df[donor_col].astype(str)
    df["is_case_sample"] = df[group_col] != control_label

    donor_is_case = df.groupby(donor_col)["is_case_sample"].any()

    donor_ids = donor_is_case.index.to_numpy()
    is_case_donor = donor_is_case.values
    is_ctrl_donor = ~is_case_donor

    donor_index_map = {pid: idx for idx, pid in enumerate(donor_ids)}
    donor_ids_all = adata.obs[donor_col].astype(str).values
    donor_inv = np.array([donor_index_map[pid] for pid in donor_ids_all], dtype=int)

    return donor_ids, donor_inv, is_case_donor, is_ctrl_donor

def compute_donor_level_odds_ratios(
    adata,
    zscore_layer=ZSCORE_LAYER,
    zscore_threshold=ZSCORE_THRESHOLD,
    group_col=GROUP_COL,
    control_label=CONTROL_LABEL,
    donor_col=DONOR_COL,
    min_donor_count=MIN_DONOR_COUNT,
    mode="lenient",
):
    if mode not in ("lenient", "strict"):
        raise ValueError("mode must be 'lenient' or 'strict'")

    donor_ids, donor_inv, is_case_donor, is_ctrl_donor = get_donor_index_and_case_status(
        adata, donor_col, group_col, control_label
    )
    n_donors = len(donor_ids)
    n_case = int(is_case_donor.sum())
    n_ctrl = int(is_ctrl_donor.sum())

    if n_case == 0 or n_ctrl == 0:
        raise ValueError("Need at least one case and one control donor.")

    print(f"[{mode}] Total donors: {n_donors}")
    print(f"[{mode}] Case donors: {n_case}, Control donors: {n_ctrl}")
    print(f"[{mode}] Total peptides: {adata.n_vars}")

    Z = adata.layers[zscore_layer]
    is_sparse = sparse.issparse(Z)

    results = []

    for j in range(adata.n_vars):
        pep_id = adata.var_names[j]

        if is_sparse:
            z_sample = Z[:, j].toarray().ravel()
        else:
            z_sample = np.asarray(Z[:, j]).ravel()

        high_sample = z_sample >= zscore_threshold
        high_int = high_sample.astype(np.int8)

        if mode == "lenient":
            high_donor_int = np.zeros(n_donors, dtype=np.int8)
            np.maximum.at(high_donor_int, donor_inv, high_int)
        else:
            high_donor_int = np.ones(n_donors, dtype=np.int8)
            np.minimum.at(high_donor_int, donor_inv, high_int)

        high_donor = high_donor_int.astype(bool)

        high_case = int((high_donor & is_case_donor).sum())
        low_case = n_case - high_case
        high_ctrl = int((high_donor & is_ctrl_donor).sum())
        low_ctrl = n_ctrl - high_ctrl

        if (min_donor_count is not None) and\
           (min(high_case, low_case, high_ctrl, low_ctrl) < min_donor_count):
            continue

        table_raw = np.array([[high_case, low_case],
                              [high_ctrl, low_ctrl]])

        if (table_raw == 0).any():
            table_for_or = table_raw + 0.5
        else:
            table_for_or = table_raw

        t_sm = Table2x2(table_for_or)
        or_est = t_sm.oddsratio

        _, p_two = fisher_exact(table_raw, alternative="two-sided")
        log_or = np.log(or_est) if or_est > 0 else np.nan

        prop_high_case = high_case / n_case
        prop_high_ctrl = high_ctrl / n_ctrl

        results.append({
            "peptide": pep_id,
            "n_high_case_donors": high_case,
            "n_case_donors": n_case,
            "n_high_control_donors": high_ctrl,
            "n_control_donors": n_ctrl,
            "prop_high_case_donors": prop_high_case,
            "prop_high_control_donors": prop_high_ctrl,
            "odds_ratio": or_est,
            "log_or": log_or,
            "p_value_two_sided": p_two,
        })

    df = pd.DataFrame(results)
    print(f"[{mode}] Computed donor-level discrete ORs for {df.shape[0]} peptides")
    return df

def compute_donor_level_logistic_ors(
    adata,
    zscore_layer=ZSCORE_LAYER,
    group_col=GROUP_COL,
    control_label=CONTROL_LABEL,
    donor_col=DONOR_COL,
    min_std=0.0,
    mode="lenient",
):
    if mode not in ("lenient", "strict"):
        raise ValueError("mode must be 'lenient' or 'strict'")

    donor_ids, donor_inv, is_case_donor, is_ctrl_donor = get_donor_index_and_case_status(
        adata, donor_col, group_col, control_label
    )
    n_donors = len(donor_ids)
    n_case = int(is_case_donor.sum())
    n_ctrl = int(is_ctrl_donor.sum())

    if n_case == 0 or n_ctrl == 0:
        raise ValueError("Need at least one case and one control donor.")

    print(f"[{mode}] Total donors: {n_donors}")
    print(f"[{mode}] Case donors: {n_case}, Control donors: {n_ctrl}")
    print(f"[{mode}] Total peptides: {adata.n_vars}")

    Z = adata.layers[zscore_layer]
    is_sparse = sparse.issparse(Z)

    y_full = is_case_donor.astype(int)
    results = []

    for j in range(adata.n_vars):
        pep_id = adata.var_names[j]

        if is_sparse:
            x_sample = Z[:, j].toarray().ravel()
        else:
            x_sample = np.asarray(Z[:, j]).ravel()

        if mode == "lenient":
            x_sample_filled = np.where(np.isnan(x_sample), -np.inf, x_sample)
            x_donor = np.full(n_donors, -np.inf, dtype=float)
            np.maximum.at(x_donor, donor_inv, x_sample_filled)
            x_donor[x_donor == -np.inf] = np.nan
        else:
            mask = ~np.isnan(x_sample)
            x_nonan = x_sample[mask]
            donor_nonan = donor_inv[mask]

            sum_per_donor = np.bincount(donor_nonan, weights=x_nonan, minlength=n_donors)
            count_per_donor = np.bincount(donor_nonan, minlength=n_donors)

            x_donor = sum_per_donor / count_per_donor
            x_donor[count_per_donor == 0] = np.nan

        mask_not_nan = ~np.isnan(x_donor)
        x = x_donor[mask_not_nan]
        y = y_full[mask_not_nan]

        if len(np.unique(y)) < 2:
            continue
        if np.nanstd(x) <= min_std:
            continue

        X = sm.add_constant(x, has_constant="add")

        try:
            model = sm.Logit(y, X)
            fit = model.fit(disp=False, maxiter=100)

            beta1 = float(fit.params[1])
            p_beta1 = float(fit.pvalues[1])

            or_est = float(np.exp(beta1))
            log_or = beta1
            p_two = p_beta1

        except Exception:
            or_est = np.nan
            log_or = np.nan
            p_two = np.nan

        results.append({
            "peptide": pep_id,
            "odds_ratio": or_est,
            "log_or": log_or,
            "p_value_two_sided": p_two,
        })

    df = pd.DataFrame(results)
    print(f"[{mode}] Computed donor-level logistic ORs for {df.shape[0]} peptides")
    return df

In [ ]:
def main(
    mode="strict",
    output_dir="results/donor_level_results",
    h5ad_path=H5AD_PATH,
    zscore_layer=ZSCORE_LAYER,
    zscore_threshold=ZSCORE_THRESHOLD,
    group_col=GROUP_COL,
    control_label=CONTROL_LABEL,
    donor_col=DONOR_COL,
    min_donor_count=MIN_DONOR_COUNT,
    min_std=0.0,
    K_values=[5, 10],
    alpha=0.05,
    label_top_n=40,
):
    os.makedirs(output_dir, exist_ok=True)
    tag = mode.lower()

    disc_csv = os.path.join(output_dir, f"donor_{tag}_peptide_discrete_OR.csv")
    logit_csv = os.path.join(output_dir, f"donor_{tag}_peptide_logistic_OR.csv")
    merged_csv = os.path.join(output_dir, f"donor_{tag}_logistic_with_counts.csv")

    adata = None
    peptide_gene_mapping = None

    def ensure_adata():
        nonlocal adata
        if adata is None:
            print("Loading AnnData...")
            adata = ad.read_h5ad(h5ad_path)
            print(f"AnnData shape: {adata.shape}")

    def ensure_mapping():
        nonlocal peptide_gene_mapping
        if peptide_gene_mapping is None:
            print("Loading peptide mapping...")
            peptide_gene_mapping = load_peptide_mapping(h5ad_path)
            print(f"Peptide mapping shape: {peptide_gene_mapping.shape}")

    if os.path.exists(disc_csv):
        print(f"\n=== [{mode}] Using existing discrete OR CSV ===")
        print(f"Loading: {disc_csv}")
        donor_peptide_disc_annot = pd.read_csv(disc_csv)
    else:
        print(f"\n=== [{mode}] Computing discrete OR ===")
        ensure_adata()
        ensure_mapping()

        donor_peptide_disc = compute_donor_level_odds_ratios(
            adata=adata,
            zscore_layer=zscore_layer,
            zscore_threshold=zscore_threshold,
            group_col=group_col,
            control_label=control_label,
            donor_col=donor_col,
            min_donor_count=min_donor_count,
            mode=mode,
        )

        donor_peptide_disc_annot = donor_peptide_disc.merge(
            peptide_gene_mapping[["seq_id", "gene", "pep_short"]],
            left_on="peptide", right_on="seq_id", how="left"
        )

        donor_peptide_disc_annot.to_csv(disc_csv, index=False)
        print(f"Saved: {disc_csv}")

    if os.path.exists(logit_csv):
        print(f"\n=== [{mode}] Using existing logistic OR CSV ===")
        print(f"Loading: {logit_csv}")
        donor_peptide_logit_annot = pd.read_csv(logit_csv)
    else:
        print(f"\n=== [{mode}] Computing logistic OR ===")
        ensure_adata()
        ensure_mapping()

        donor_peptide_logit = compute_donor_level_logistic_ors(
            adata=adata,
            zscore_layer=zscore_layer,
            group_col=group_col,
            control_label=control_label,
            donor_col=donor_col,
            min_std=min_std,
            mode=mode,
        )

        donor_peptide_logit_annot = donor_peptide_logit.merge(
            peptide_gene_mapping[["seq_id", "gene", "pep_short"]],
            left_on="peptide", right_on="seq_id", how="left"
        )

        donor_peptide_logit_annot.to_csv(logit_csv, index=False)
        print(f"Saved: {logit_csv}")

In [ ]:
main(
    mode="strict",
    output_dir="results/donor_level_results_strict",
    K_values=[5, 10],
    alpha=0.05,
    label_top_n=40,
)